# 01 — Data Preparation
Prepare the invoice extraction dataset: load existing data, generate synthetic examples, merge, and format for training.

In [ ]:
!pip install -q transformers datasets pydantic openai rapidfuzz python-dotenv Pillow

In [ ]:
import os
import sys
from dotenv import load_dotenv

sys.path.insert(0, "..")
load_dotenv()

from src.data.schema import Invoice, LineItem
from src.data.existing_loader import load_existing_dataset
from src.data.synthetic_gen import generate_dataset
from src.data.merge import merge_and_split, deduplicate
from src.data.format import format_dataset, save_jsonl

## Step 1: Load Existing Dataset

In [ ]:
# Load and normalize existing HF dataset (returns (ocr_text, Invoice) pairs)
existing_pairs = load_existing_dataset(
    dataset_name="mychen76/invoices-and-receipts_ocr_v1",
    max_samples=500,
)
print(f"Loaded {len(existing_pairs)} existing invoice pairs")
if existing_pairs:
    text, invoice = existing_pairs[0]
    print("\n=== OCR Text (first 500 chars) ===")
    print(text[:500])
    print("\n=== Extracted Fields ===")
    print(invoice.model_dump_json(indent=2))

## Step 2: Generate Synthetic Dataset

In [ ]:
synthetic_data = generate_dataset(total=1500, batch_size=5)
print(f"Generated {len(synthetic_data)} synthetic invoices")
if synthetic_data:
    text, invoice = synthetic_data[0]
    print("=== Invoice Text ===")
    print(text[:500])
    print("\n=== Extracted Fields ===")
    print(invoice.model_dump_json(indent=2))

## Step 3: Merge and Split

In [ ]:
# existing_pairs is already list[tuple[str, Invoice]] from load_existing_dataset
train_data, eval_data = merge_and_split(existing_pairs, synthetic_data, eval_size=500, seed=42)
print(f"Train set: {len(train_data)} examples")
print(f"Eval set: {len(eval_data)} examples")

## Step 4: Format and Save

In [ ]:
import json
train_formatted = format_dataset(train_data)
eval_formatted = format_dataset(eval_data)

os.makedirs("../data", exist_ok=True)
save_jsonl(train_formatted, "../data/train.jsonl")
save_jsonl(eval_formatted, "../data/eval.jsonl")

print(f"Saved {len(train_formatted)} training examples to data/train.jsonl")
print(f"Saved {len(eval_formatted)} eval examples to data/eval.jsonl")
print("\n=== Formatted Example ===")
print(json.dumps(train_formatted[0], indent=2)[:1000])